# BaCP: verify, then run

Run the cells top to bottom. The verification ladder comes first on purpose: if a
level fails, every experiment below it is untrustworthy, and the failing level tells
you where to look.

Levels depend only on the levels below them, so when several fail at once, **fix the
lowest one first**:

| Level | Covers |
|---|---|
| L0 / L0b | imports, undefined names, star-import hazards; model + dataset registries |
| L1 / L1b | loss math; the CAP-aligned loss |
| L2 | augmentation views, collate shapes |
| L3 | model wrapper, weight loading |
| L4 / L5 | pruner masks; sparsity accounting |
| L6a / L6b / L6c | DyReLU phasing; weight sharing; EAST cyclic sparsity |
| L7 | the eight initialisers, one optimizer step |
| L8 / L9 | baseline loop; full BaCP loop |

Locally the same thing runs via `.\run_tests.ps1` (fast, ~10s) or
`.\run_tests.ps1 -Full`.

In [ ]:
# Fast suite: no GPU, no downloads, ~10s. Run after every edit.
!cd ../.. && python -m pytest -m "not slow" -q

In [ ]:
# Full suite: adds the real training loops (L8, L9). ~15s.
!cd ../.. && python -m pytest -q

In [ ]:
# One level at a time while debugging, e.g. the EAST pruner:
!cd ../.. && python -m pytest project/tests/test_east_pruner.py -v

## Read this before running anything

### 1. What "sparsity" now measures changed

`layer_check` previously marked **word embeddings** as prunable. On DistilBERT that
is a single 23.4M-parameter tensor out of 66M, and DistilBERT *ties* it to the output
projection -- so pruning the embedding matrix also pruned the LM head. The prunable
set was 99.7% of all parameters.

Appendix D.4 of the paper says "embeddings and classification heads were kept dense".
The code now does that:

| | before | after |
|---|---|---|
| prunable share (DistilBERT) | 99.7% | **63.3%** (36 transformer linears) |
| embeddings | pruned | excluded |
| LM / classifier head | pruned | excluded |
| `cls_head` (vision) | pruned | excluded |

**Every sparsity number in the paper predates this.** The vision change is not
cosmetic either: at 99% sparsity on ResNet-50/CIFAR-100 a dense `Linear(2048, 100)`
is 204,800 weights against a surviving budget of roughly 250,000.

### 2. Check the checkpoint paths

They are all `/dbfs` paths, and `load_weights` returns False rather than raising when
one is missing -- so a run can train from random init while looking like it loaded
pretrained weights. Every dense baseline in the paper sits 1.7-4.9 points below the
published number for the same architecture and dataset, which is what you would
expect from that plus a 15-epoch budget.

### 3. Use the canonical protocol

The GraNet/GraSP/RigL benchmark lineage is 160 epochs, batch 128, SGD lr 0.1 with
`÷10 at [80, 120]`, weight decay 5e-4, 3 seeds. The current defaults are 5+10 epochs
at batch 512 with no LR schedule, which is 10-16x short and accounts for the
baseline gap above.

In [ ]:
import os

BASELINE = '/dbfs/research/bacp/resnet50/cifar100/resnet50_cifar100_dense_baseline.pt'
IMAGENET = '/dbfs/research/bacp/resnet50_imagenet.pth'

for path in (BASELINE, IMAGENET):
    print(('FOUND  ' if os.path.exists(path) else 'MISSING'), path)

# The canonical CIFAR pruning protocol. Everything below inherits it.
PROTOCOL = (
    '--epochs 160 --batch_size 128 --optimizer_type sgd --learning_rate 0.1 '
    '--scheduler_type cosine --recovery_epochs 10'
)

COMMON = (
    '--model_name resnet50 --model_type cv --dataset_name cifar100 --num_classes 100 '
    f'--trained_weights {BASELINE} '
    '--pruning_type magnitude --target_sparsity 0.99 --sparsity_scheduler cubic '
    f'{PROTOCOL} --enable_finetune --databricks_env --log_to_wandb --log_epochs'
)
print()
print(COMMON)

## Experiment 1 - the CE-only control

**The single highest-value run.** BaCP's exact recipe with the three contrastive
weights zeroed: same schedule, same augmentation, same epoch budget, cross-entropy
only.

It answers the question the paper currently cannot: are the gains from the
contrastive objective, or from the training recipe around it? Run this first, because
it decides how everything else is framed.

`--lambdas` is ordered **PrC FiC SnC CE**, matching `_combine_losses`.

In [ ]:
!python ../scripts/bacp_script.py {COMMON} --lambdas 0 0 0 1 --experiment_type control_ce_only

## Experiment 2 - legacy vs CAP-aligned

The central claim after this work: transferring CAP's formulation correctly is what
makes the contrastive terms bite.

- `legacy` + `current` reproduces the published behaviour (SupCon + NT-Xent summed
  over a square 2Bx2B matrix, per-model trainable heads).
- `cap` + `tied_frozen` is one CAP Eq.1 functional over rectangular
  student-anchor/teacher-candidate similarities, with a frozen shared head.

Keep both. A reported before/after is far stronger than quietly replacing the table.

In [ ]:
!python ../scripts/bacp_script.py {COMMON} --contrastive_mode legacy --proj_mode current --experiment_type ab_legacy

In [ ]:
!python ../scripts/bacp_script.py {COMMON} --contrastive_mode cap --proj_mode tied_frozen --experiment_type ab_cap_aligned

## Experiment 3 - module ablation, where there is headroom

The published ablation ran on ResNet-50/CIFAR-10, which spans 92-93.5% no matter what
you do -- which is why every component looked useless. CIFAR-100 @ 0.99 has ~8 points
to measure; the ViT and RoBERTa settings below have far more.

One weight zeroed at a time, in `_combine_losses` order (PrC FiC SnC CE).

In [ ]:
ABLATIONS = {
    'full':   '0.25 0.25 0.25 0.25',
    'no_prc': '0 0.33 0.33 0.33',
    'no_fic': '0.33 0 0.33 0.33',
    'no_snc': '0.33 0.33 0 0.33',
    'no_ce':  '0.33 0.33 0.33 0',
}
for name, lam in ABLATIONS.items():
    print('--- ' + name + ' ---')
    !python ../scripts/bacp_script.py {COMMON} --lambdas {lam} --experiment_type abl_{name}

## Experiment 4 - Vision Transformers

Now runnable: `vit-tiny` and `vit-small` are registered and verified (construction and
forward pass both checked on CPU).

Three things to know:

- **`--image_size 224`.** These are `patch16-224` checkpoints; 32x32 inputs will not
  tokenise. Batch size drops accordingly.
- **No DyReLU.** The builders raise if you pass `--dyrelu_en` or
  `--dyrelu_phasing_en`, rather than silently training a plain-ReLU network. DyReLU is
  threaded into this project's own ResNet/VGG blocks and has no HF equivalent.
- **No published baseline exists** for ViT at 95-99% *unstructured* sparsity on CIFAR.
  Published ViT pruning is structured (SAViT, X-Pruner, DIMAP) or stops around 80-90%.
  That makes this a genuine contribution *and* means you have nothing to beat -- say
  so explicitly rather than leaving a reviewer to wonder.

ViT-Small/CIFAR-100 @99% is the widest gap in the paper (dense 90.85 vs I.P. 31.91),
so it is the best ablation setting available.

In [ ]:
VIT = (
    '--model_name vit-small --model_type cv --dataset_name cifar100 --num_classes 100 '
    '--image_size 224 --batch_size 64 '
    '--trained_weights /dbfs/research/bacp/vit-small/cifar100/vit_small_cifar100_baseline.pt '
    '--pruning_type magnitude --target_sparsity 0.99 --sparsity_scheduler cubic '
    '--epochs 50 --recovery_epochs 10 --optimizer_type adamw --learning_rate 1e-4 '
    '--enable_finetune --databricks_env --log_to_wandb'
)

!python ../scripts/bacp_script.py {VIT} --experiment_type vit_small_cap

In [ ]:
# The module ablation in the highest-headroom vision setting.
for name, lam in ABLATIONS.items():
    print('--- vit ' + name + ' ---')
    !python ../scripts/bacp_script.py {VIT} --lambdas {lam} --experiment_type vit_abl_{name}

## Experiment 5 - language models

Also now runnable, and verified end-to-end: a full BaCP contrastive step on
DistilBERT/SST-2 produces `PrC 2.26 / CE 0.70`, with gradient reaching the last
transformer layer and **exactly zero** reaching the frozen projection head.

- `--model_type llm` selects the text pipeline (salvaged out of the orphaned
  `dataset_utils_old.py`, which nothing imported -- so the SST-2 and WikiText-2 halves
  of the results table previously had no reachable code path).
- SST-2 uses the GLUE **validation** split for evaluation, since the test split is
  unlabelled. Every paper in this space does the same; CAP reports "results on
  development sets".
- WikiText-2 is masked language modelling: `DataCollatorForLanguageModeling` at 15%
  masking, and accuracy is scored over masked positions only.
- `--num_classes` is required by the parser but ignored for MLM.
- Use AdamW at 2e-5, not SGD at 0.1.

CAP's tuned temperature is **0.1** (swept over {0.05, 0.1, 0.2, 0.3}); the default
here is 0.07.

In [ ]:
LM_COMMON = (
    '--model_type llm --batch_size 32 --optimizer_type adamw --learning_rate 2e-5 '
    '--optimizer_type_ft adamw --learning_rate_ft 2e-5 --tau 0.1 '
    '--pruning_type magnitude --sparsity_scheduler cubic '
    '--epochs 10 --recovery_epochs 5 --enable_finetune --databricks_env --log_to_wandb'
)

# SST-2, sequence classification
for model in ('distilbert-base-uncased', 'roberta-base'):
    for sparsity in (0.95, 0.97, 0.99):
        print(f'--- {model} sst2 @ {sparsity} ---')
        !python ../scripts/bacp_script.py {LM_COMMON}             --model_name {model} --dataset_name sst2 --num_classes 2             --target_sparsity {sparsity}             --trained_weights /dbfs/research/bacp/{model}/sst2/baseline.pt             --experiment_type lm_sst2_{sparsity}

In [ ]:
# WikiText-2, masked language modelling. --num_classes is ignored here.
for model in ('distilbert-base-uncased-mlm', 'roberta-base-mlm'):
    for sparsity in (0.95, 0.97, 0.99):
        print(f'--- {model} wikitext2 @ {sparsity} ---')
        !python ../scripts/bacp_script.py {LM_COMMON}             --model_name {model} --dataset_name wikitext2 --num_classes 2             --target_sparsity {sparsity}             --trained_weights /dbfs/research/bacp/{model}/wikitext2/baseline.pt             --experiment_type lm_mlm_{sparsity}

## Experiment 6 - three seeds

Every ablation delta in the paper is under 0.4% with no error bars, and the
reproducibility checklist answers `yes` to variance analysis. Neither is defensible
without this. A delta smaller than the seed spread is not a finding.

In [ ]:
for seed in (42, 43, 44):
    print('--- seed ' + str(seed) + ' ---')
    !python ../scripts/bacp_script.py {COMMON} --seed {seed} --experiment_type seed_{seed}

## Experiment 7 - proj_mode ablation

Cheap, and direct evidence for the projection-head argument. Under `current` the head
is a trainable, unpruned 262k-parameter adapter that the contrastive losses can
satisfy without touching the backbone; `tied_frozen` and `none` both remove that route.

`test_projection_head_cannot_absorb_the_objective` already proves the gradient claim
on a tiny model, and the DistilBERT run above confirms it at scale -- this measures
whether it changes accuracy.

In [ ]:
for mode in ('current', 'tied_frozen', 'none'):
    print('--- proj_mode=' + mode + ' ---')
    !python ../scripts/bacp_script.py {COMMON} --proj_mode {mode} --experiment_type proj_{mode}

## Experiment 8 - EAST

**EAST has never actually run.** `EASTPruner` could not be constructed (`Tc_ratio` was
never passed), and even once constructed its `update_masks` re-gated on
`current_idx % delta_T` after `BasePruner.step` had already added 1 -- so it returned
immediately every time and the masks stayed frozen at their Bernoulli initialisation.
A third bug then surfaced the moment it did run: `k_grow` is derived from the global
sparsity ratio but spent per-layer, so a layer below the average ERK density was asked
to regrow more connections than it had inactive slots.

These are therefore first results, not a reproduction.

Two parameterisation notes from the paper (arXiv 2411.13545):

- Cyclic sparsity is parameterised by a **density multiplier m = 10x**, not an
  absolute floor: `s_min = 1 - m(1 - s_max)`. At `s_max=0.99` that is `s_min=0.90`,
  not the 0.05 default -- which at 99% target would be a 95x multiplier.
- EAST uses **2 cycles**, a 2x length multiplier, and `delta_T = 4000` on CIFAR.

In [ ]:
S_MAX, M = 0.99, 10
S_MIN = 1 - M * (1 - S_MAX)      # 0.90
print(f's_max={S_MAX}  m={M}x  ->  s_min={S_MIN:.2f}')

EAST_COMMON = (COMMON.replace('--pruning_type magnitude', '--pruning_type east')
                     .replace('--sparsity_scheduler cubic', '--sparsity_scheduler cyclic'))

!python ../scripts/bacp_script.py {EAST_COMMON} --delta_T 4000 --experiment_type east_cyclic

In [ ]:
!python ../scripts/bacp_script.py {COMMON} --weight_sharing_en --experiment_type east_sharing

In [ ]:
# DyReLU phasing is vision-only and works on this project's ResNet/VGG blocks.
!python ../scripts/bacp_script.py {COMMON} --dyrelu_phasing_en --experiment_type east_dyrelu

## Inspect a finished run

Reported sparsity is sparsity **of the prunable set**, which after the `layer_check`
correction excludes 1-D parameters, embeddings, task heads, the contrastive projection
head, and the DyReLU hyperfunctions. Under weight sharing it is also computed over
*unique* parameters, since `named_parameters()` de-duplicates aliased tensors.

All of those are the right denominators -- just state in the paper which one you are
reporting, because the same checkpoint yields different numbers under different
conventions.

In [ ]:
import sys
sys.path.append('..')

from model_factory import ClassificationAndEncoderNetwork
from pruning_factory import check_sparsity_distribution, layer_check
from utils import load_weights

CHECKPOINT = '/dbfs/research/bacp/resnet50/cifar100/<date>/<file>.pt'

model = ClassificationAndEncoderNetwork(
    model_name='resnet50', num_classes=100, num_out_features=128,
    device='cuda', adapt=True, pretrained=False,
)
if load_weights(model, CHECKPOINT):
    check_sparsity_distribution(model)

    prunable = sum(p.numel() for n, p in model.named_parameters() if layer_check(n, p))
    total = sum(p.numel() for p in model.parameters())
    print(f'\nprunable {prunable:,} / {total:,} = {prunable/total:.1%} of all params')
else:
    print('Checkpoint not loaded - anything below would describe a random model.')